# Train on Kaggle

Settings (right sidebar): **Accelerator: GPU T4**, **Internet: on**.
Use "Save Version" -> "Save & Run All" so it keeps running with the tab closed.
The trained weights end up in the Output tab, named `<model>-<params>-c<context>-d<data>.pt`.

In [ ]:
MODEL = "gpt"                   # bigram, mlp, gpt (10.8M) or gpt-20m
DATASET = "tinystories-train"   # tinystories (5M valid split) or tinystories-train
MAX_CHARS = 50_000_000
BLOCK_SIZE = 512                # context length, the batch size shrinks to keep characters per step the same
MAX_ITERS = 5000
AMP = False                     # True = fp16 mixed precision (faster), adds -fp16 to the run name

In [ ]:
import os

os.chdir("/kaggle/working")
if not os.path.exists("gecko"):
    !git clone --depth 1 https://github.com/focuzd/gecko.git
os.chdir("gecko")
!git log --oneline -1

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU, check the accelerator setting")

In [ ]:
amp_flag = "--amp" if AMP else ""
!python -u src/train.py --model {MODEL} --dataset {DATASET} --max-chars {MAX_CHARS} --block-size {BLOCK_SIZE} --max-iters {MAX_ITERS} --resume {amp_flag}

In [ ]:
from pathlib import Path

weights = max(Path("weights").glob("*.pt"), key=lambda p: p.stat().st_mtime)  # the run that just finished
print(weights)
!python -u src/evaluate.py {weights}
!python -u src/sample_stories.py --weights {weights} --num-stories 3 --seed 42

In [ ]:
# keep the output small: just the weights and the sampled stories
!cp {weights} outputs/{weights.stem}_*.txt /kaggle/working/
!rm -f data/*.txt
!ls -lh /kaggle/working